# Deep dive 11/13: `digital_events`

Dictionary declares 10,000,000 rows, Source: Digital Banking, Partition: daily -- the **largest fact table** in the dataset (per the dictionary). Note a discrepancy worth checking directly: `table_specs.py`'s own comment above `digital_events_spec` calls it "fact, 15.6M rows", while the dictionary says 10,000,000 -- section 2 checks what the real Bronze row count actually is, since neither number can be assumed correct given the project's established pattern of unreliable dictionary counts.

25 columns total. Dictionary NOT NULL: `event_id`, `event_date`, `process_date`, `session_id`, `event_type`, `event_category`, `channel`, `is_mobile` -- **8 columns**. `contracts.py`'s current `required` tuple has 7 -- **`is_mobile` is missing**, checked in section 3 as a fix candidate. `customer_id` is a nullable FK here (unlike e.g. `complaints.customer_id`, which is FK NOT NULL) -- not every digital event needs to be tied to an identified customer (anonymous/pre-login browsing), so this is expected, not a gap.

`ip_country` uses the same `country` transform as `transactions.transaction_country` -- the one that canonicalizes the real `México`/`Mexico` spelling split. Section 7 checks whether `ip_country` needed and got the same fix.

This is a very large table (potentially 10-15M rows) -- no full-table dumps, aggregate queries only, sample rows by ID for the row-level look.

Run top to bottom. Checklist near the end; free cells after that.

In [45]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side

In [46]:
sample_ids = q("SELECT event_id FROM bronze.digital_events ORDER BY event_id LIMIT 20")["event_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = ("event_id, event_date, process_date, customer_id, session_id, event_type, event_category, "
        "channel, platform, browser, action, product_id, event_value, duration_seconds, "
        "ip_country, is_mobile, utm_source, utm_medium, utm_campaign")
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.digital_events WHERE event_id IN ({ids_sql}) ORDER BY event_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_digital_events WHERE event_id IN ({ids_sql}) ORDER BY event_id"))

Bronze:


,event_id,event_date,process_date,customer_id,session_id,event_type,event_category,channel,platform,browser,action,product_id,event_value,duration_seconds,ip_country,is_mobile,utm_source,utm_medium,utm_campaign
0,EVT-00004OPNGBI2S1FVLZNQ,2025-07-09 11:59:17,2025-07-09,CLI-70N5OYJK472H,SES-POA320SNR9ZYF87WS67UDZAEVTEM,PageView,Transaction,Desktop Web,Linux,Chrome,NaN,None,NaN,280.0,Argentina,False,None,None,None
1,EVT-0000L3189EFFIS2H470M,2023-10-27 03:16:00,2023-10-26,CLI-6OJ56FJ5A8Z5,SES-GX7MLH01BSEEMOD2V5B2AK8JA4DE,Click,Navigation,Android App,Android,NaN,NaN,None,NaN,NaN,Argentina,True,None,None,None
2,EVT-0000L9PLFE1UHCAMI1UJ,2025-02-28 02:20:47,2025-02-27,NaN,SES-KYQNT68GKI2JNZVLB6A86LX6TXAQ,PageView,Navigation,Android App,Android,NaN,view_help,None,NaN,219.0,Argentina,True,None,None,None
3,EVT-0000PVZ0BKDHZ4HDEEUG,2024-03-03 20:43:38,2024-03-03,CLI-R267B0HLL489,SES-IQ06O451ZPN57NKJ405AK159AI3U,Logout,Authentication,iOS App,iOS,NaN,login,None,NaN,NaN,México,True,None,None,None
4,EVT-0000QZ4U3WO4IBJU2T27,2023-07-06 23:52:46,2023-07-06,CLI-T8ZBJU9H8EQ4,SES-VRHRX5UC8YMJXEEI0I72FKGD0CJJ,PageView,Transaction,Desktop Web,Linux,Edge,initiate_transfer,None,NaN,280.0,México,False,None,None,None
5,EVT-00011BNKAZ31I76XDD3E,2024-09-30 11:17:12,2024-09-30,CLI-0I14J9OON4GS,SES-XPS9VYVDMXKWJIZG63KPRGUOSG5V,Click,Navigation,Android App,Android,NaN,view_home,None,NaN,NaN,México,True,None,None,None
6,EVT-00011GBHT46ABRV68389,2023-10-24 00:05:19,2023-10-23,NaN,SES-L790WN061RVSET26HX630836UQGI,Logout,Authentication,Mobile Web,iOS,Samsung Internet,login,None,NaN,NaN,Mexico,True,None,None,None
7,EVT-00011QDKHT5KHXRDW7AB,2026-06-13 01:37:23,2026-06-12,CLI-5AXY24603TPE,SES-GHYV9BKEF6HPFOW45XE64YNXB8IL,Click,Navigation,iOS App,iOS,NaN,view_products,None,NaN,NaN,Colombia,True,None,None,None
8,EVT-00018SPQDKIFWZXG5IO1,2026-02-10 11:02:32,2026-02-10,CLI-EQYLDTK08QWP,SES-B6JUOC4JHTHWWTO529V64PU165PS,PageView,Product,Android App,Android,NaN,view_product,None,NaN,NaN,Argentina,True,None,None,None
9,EVT-0001BK1GSWT8B7OUFHF8,2024-06-27 05:50:28,2024-06-26,CLI-2U5WTORAP6XJ,SES-FCETKEAA83FJVWK5AEDF8T1P1RZ3,Logout,Authentication,Android App,Android,NaN,login,None,NaN,NaN,Colombia,True,None,None,None


Silver:


,event_id,event_date,process_date,customer_id,session_id,event_type,event_category,channel,platform,browser,action,product_id,event_value,duration_seconds,ip_country,is_mobile,utm_source,utm_medium,utm_campaign
0,EVT-00004OPNGBI2S1FVLZNQ,2025-07-09 11:59:17,2025-07-09,CLI-70N5OYJK472H,SES-POA320SNR9ZYF87WS67UDZAEVTEM,PageView,Transaction,Desktop Web,Linux,Chrome,NaN,None,NaN,280.0,Argentina,False,None,None,None
1,EVT-0000L3189EFFIS2H470M,2023-10-27 03:16:00,2023-10-26,CLI-6OJ56FJ5A8Z5,SES-GX7MLH01BSEEMOD2V5B2AK8JA4DE,Click,Navigation,Android App,Android,NaN,NaN,None,NaN,NaN,Argentina,True,None,None,None
2,EVT-0000L9PLFE1UHCAMI1UJ,2025-02-28 02:20:47,2025-02-27,NaN,SES-KYQNT68GKI2JNZVLB6A86LX6TXAQ,PageView,Navigation,Android App,Android,NaN,view_help,None,NaN,219.0,Argentina,True,None,None,None
3,EVT-0000PVZ0BKDHZ4HDEEUG,2024-03-03 20:43:38,2024-03-03,CLI-R267B0HLL489,SES-IQ06O451ZPN57NKJ405AK159AI3U,Logout,Authentication,iOS App,iOS,NaN,login,None,NaN,NaN,México,True,None,None,None
4,EVT-0000QZ4U3WO4IBJU2T27,2023-07-06 23:52:46,2023-07-06,CLI-T8ZBJU9H8EQ4,SES-VRHRX5UC8YMJXEEI0I72FKGD0CJJ,PageView,Transaction,Desktop Web,Linux,Edge,initiate_transfer,None,NaN,280.0,México,False,None,None,None
5,EVT-00011BNKAZ31I76XDD3E,2024-09-30 11:17:12,2024-09-30,CLI-0I14J9OON4GS,SES-XPS9VYVDMXKWJIZG63KPRGUOSG5V,Click,Navigation,Android App,Android,NaN,view_home,None,NaN,NaN,México,True,None,None,None
6,EVT-00011GBHT46ABRV68389,2023-10-24 00:05:19,2023-10-23,NaN,SES-L790WN061RVSET26HX630836UQGI,Logout,Authentication,Mobile Web,iOS,Samsung Internet,login,None,NaN,NaN,México,True,None,None,None
7,EVT-00011QDKHT5KHXRDW7AB,2026-06-13 01:37:23,2026-06-12,CLI-5AXY24603TPE,SES-GHYV9BKEF6HPFOW45XE64YNXB8IL,Click,Navigation,iOS App,iOS,NaN,view_products,None,NaN,NaN,Colombia,True,None,None,None
8,EVT-00018SPQDKIFWZXG5IO1,2026-02-10 11:02:32,2026-02-10,CLI-EQYLDTK08QWP,SES-B6JUOC4JHTHWWTO529V64PU165PS,PageView,Product,Android App,Android,NaN,view_product,None,NaN,NaN,Argentina,True,None,None,None
9,EVT-0001BK1GSWT8B7OUFHF8,2024-06-27 05:50:28,2024-06-26,CLI-2U5WTORAP6XJ,SES-FCETKEAA83FJVWK5AEDF8T1P1RZ3,Logout,Authentication,Android App,Android,NaN,login,None,NaN,NaN,Colombia,True,None,None,None


## 1. Column inventory -- names AND types, Bronze vs. Silver (all 25 columns)

In [47]:
expected_silver_columns = [
    "event_id", "event_date", "process_date", "customer_id", "session_id", "event_type",
    "event_category", "channel", "platform", "browser", "app_version", "page_url", "page_title",
    "action", "element_id", "product_id", "event_value", "duration_seconds", "ip_address",
    "ip_country", "ip_city", "is_mobile", "referrer", "utm_source", "utm_medium", "utm_campaign",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_digital_events'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_digital_events:", missing or "none")
print("Extra in silver.fact_digital_events:", extra or "none")
print()
print(f"Total expected: {len(expected_silver_columns)}, total actual: {len(actual_silver_columns)}")
print("Column order as built:", actual_silver_columns)

Missing from silver.fact_digital_events: none
Extra in silver.fact_digital_events: none

Total expected: 26, total actual: 26
Column order as built: ['event_id', 'event_date', 'process_date', 'customer_id', 'session_id', 'event_type', 'event_category', 'channel', 'platform', 'browser', 'app_version', 'page_url', 'page_title', 'action', 'element_id', 'product_id', 'event_value', 'duration_seconds', 'ip_address', 'ip_country', 'ip_city', 'is_mobile', 'referrer', 'utm_source', 'utm_medium', 'utm_campaign']


In [48]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='digital_events' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_digital_events' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,event_id,VARCHAR
1,event_date,VARCHAR
2,process_date,VARCHAR
3,customer_id,VARCHAR
4,session_id,VARCHAR
5,event_type,VARCHAR
6,event_category,VARCHAR
7,channel,VARCHAR
8,platform,VARCHAR
9,browser,VARCHAR


Silver columns + types:


,column_name,data_type
0,event_id,VARCHAR
1,event_date,TIMESTAMP
2,process_date,DATE
3,customer_id,VARCHAR
4,session_id,VARCHAR
5,event_type,VARCHAR
6,event_category,VARCHAR
7,channel,VARCHAR
8,platform,VARCHAR
9,browser,VARCHAR


## 2. Row-count reconciliation + dedup

Checking both candidate numbers: the dictionary's declared 10,000,000 AND `table_specs.py`'s comment claiming 15.6M.

In [49]:
bronze_total = q("SELECT count(*) AS n FROM bronze.digital_events")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT event_id) AS n FROM bronze.digital_events")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_digital_events")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT event_id) AS n FROM silver.fact_digital_events")["n"][0]

print(f"bronze.digital_events:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct event_id")
print(f"silver.fact_digital_events: {silver_total:,} rows, {silver_distinct_pk:,} distinct event_id")
print()
print(f"Dictionary declares 10,000,000 rows -- {'matches' if bronze_total == 10_000_000 else f'MISMATCH, off by {abs(bronze_total-10_000_000):,}'}")
print(f"table_specs.py comment claims 15.6M rows -- {'matches' if abs(bronze_total-15_600_000) < 50_000 else f'MISMATCH, off by {abs(bronze_total-15_600_000):,}'}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.digital_events:      15,620,994 rows, 15,620,994 distinct event_id
silver.fact_digital_events: 15,620,994 rows, 15,620,994 distinct event_id

Dictionary declares 10,000,000 rows -- MISMATCH, off by 5,620,994
table_specs.py comment claims 15.6M rows -- matches
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 8 dictionary-declared columns

Including `is_mobile`, missing from `contracts.py`'s `required` tuple -- checked here as a fix candidate.

In [50]:
declared_not_null = ["event_id", "event_date", "process_date", "session_id", "event_type",
                      "event_category", "channel", "is_mobile"]
currently_required = {"event_id","event_date","process_date","session_id","event_type","event_category","channel"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.digital_events
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_digital_events WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,column,declared,in_contracts_required,bronze_nulls,silver_nulls,cast_introduced_nulls
0,event_id,NOT NULL,True,0,0,0
1,event_date,NOT NULL,True,0,0,0
2,process_date,NOT NULL,True,0,0,0
3,session_id,NOT NULL,True,0,0,0
4,event_type,NOT NULL,True,0,0,0
5,event_category,NOT NULL,True,0,0,0
6,channel,NOT NULL,True,0,0,0
7,is_mobile,NOT NULL,False,0,0,0


## 4. Nullable columns -- population rate, Bronze vs. Silver (all 17 nullable columns)

In [51]:
nullable_cols = ["customer_id", "platform", "browser", "app_version", "page_url", "page_title",
                  "action", "element_id", "product_id", "event_value", "duration_seconds",
                  "ip_address", "ip_country", "ip_city", "referrer", "utm_source", "utm_medium",
                  "utm_campaign"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.digital_events")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.digital_events
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_digital_events WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

,column,total,bronze_populated,bronze_pct,silver_populated,silver_pct
0,customer_id,15620994,11875548,76.02,11875548,76.02
1,platform,15620994,14840467,95.00,14840467,95.00
2,browser,15620994,5933258,37.98,5933258,37.98
3,app_version,15620994,8906578,57.02,8906578,57.02
4,page_url,15620994,14840600,95.00,14840600,95.00
5,page_title,15620994,14841170,95.01,14841170,95.01
6,action,15620994,14059562,90.00,14059562,90.00
7,element_id,15620994,13277750,85.00,13277750,85.00
8,product_id,15620994,1440338,9.22,1440338,9.22
9,event_value,15620994,794510,5.09,794510,5.09


## 5. Domain checks -- `event_type`, `event_category`, `channel`, `platform`

Dictionary text for `event_type`/`event_category` is truncated in the PDF (overlapping "NOT NULL" text), so pulling the real values directly rather than assuming the dictionary's partial list is complete.

In [52]:
# event_type -- dictionary text (partial/truncated): PageView, Click, FormSubmit, Login, Logout, Error, ...
print("event_type:")
display(q("SELECT event_type, count(*) AS n FROM bronze.digital_events GROUP BY 1 ORDER BY n DESC"))
# event_category -- dictionary text (partial/truncated): Navigation, Transaction, Authentication, Product, ...
print("event_category:")
display(q("SELECT event_category, count(*) AS n FROM bronze.digital_events GROUP BY 1 ORDER BY n DESC"))
# channel -- dictionary: Android App, iOS App, Desktop Web, Mobile Web
print("channel:")
display(q("SELECT channel, count(*) AS n FROM bronze.digital_events GROUP BY 1 ORDER BY n DESC"))
# platform -- dictionary: Android, iOS, Windows, MacOS, Linux
print("platform:")
display(q("SELECT platform, count(*) AS n FROM bronze.digital_events GROUP BY 1 ORDER BY n DESC"))

event_type:


,event_type,n
0,PageView,5972564
1,Click,3585034
2,Login,2434770
3,Logout,2433612
4,FormSubmit,596908
5,Error,358723
6,Purchase,239383


event_category:


,event_category,n
0,Authentication,4868382
1,Navigation,3961324
2,Product,3786314
3,Transaction,3004974


channel:


,channel,n
0,Android App,5476164
1,iOS App,3899497
2,Desktop Web,3128851
3,Mobile Web,3116482


platform:


,platform,n
0,Android,6685963
1,iOS,5182421
2,Windows,991807
3,Linux,991774
4,MacOS,988502
5,NaN,780527


## 6. FK integrity -- `customer_id` (nullable), `product_id` (nullable), reproduced directly

In [53]:
fk_checks = [
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("product_id", "silver.dim_products", "product_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_digital_events WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_digital_events t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

,fk,populated,orphans,orphan_pct
0,customer_id -> silver.dim_customers.customer_id,11875548,0,0.0
1,product_id -> silver.dim_products.product_id,1440338,0,0.0


## 7. `ip_country` -- canonicalization check

Same `country` transform used on `transactions.transaction_country`, which had a real `México`/`Mexico` spelling split in Bronze that Silver's `COUNTRY_CANONICAL` mapping fixes. Checking whether `ip_country` had (and got) the same fix.

In [54]:
print("Bronze ip_country (raw, pre-canonicalization):")
display(q("SELECT ip_country, count(*) AS n FROM bronze.digital_events GROUP BY 1 ORDER BY n DESC"))
print("Silver ip_country (post-canonicalization):")
display(q("SELECT ip_country, count(*) AS n FROM silver.fact_digital_events GROUP BY 1 ORDER BY n DESC"))

Bronze ip_country (raw, pre-canonicalization):


,ip_country,n
0,México,6242893
1,Colombia,4806882
2,Argentina,3533045
3,Mexico,1038174


Silver ip_country (post-canonicalization):


,ip_country,n
0,México,7281067
1,Colombia,4806882
2,Argentina,3533045


## 8. `is_mobile` -- boolean sanity, cross-checked against `channel`

In [55]:
print("is_mobile value counts (Silver, typed BOOLEAN):")
display(q("SELECT is_mobile, count(*) AS n FROM silver.fact_digital_events GROUP BY 1 ORDER BY 1"))
print()
print("is_mobile vs. channel -- does the flag line up with Android App/iOS App/Mobile Web vs. Desktop Web?")
display(q("""
    SELECT channel, is_mobile, count(*) AS n
    FROM silver.fact_digital_events
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

is_mobile value counts (Silver, typed BOOLEAN):


,is_mobile,n
0,False,3128851
1,True,12492143



is_mobile vs. channel -- does the flag line up with Android App/iOS App/Mobile Web vs. Desktop Web?


,channel,is_mobile,n
0,Android App,True,5476164
1,Desktop Web,False,3128851
2,Mobile Web,True,3116482
3,iOS App,True,3899497


## 9. `event_value`/`duration_seconds` -- numeric range sanity

In [56]:
q("""
    SELECT
        typeof(event_value) AS event_value_type, typeof(duration_seconds) AS duration_type,
        min(event_value) AS min_value, max(event_value) AS max_value,
        min(duration_seconds) AS min_duration, max(duration_seconds) AS max_duration,
        count(*) FILTER (WHERE event_value < 0) AS negative_event_value,
        count(*) FILTER (WHERE duration_seconds < 0) AS negative_duration
    FROM silver.fact_digital_events
    GROUP BY ALL
""")

,event_value_type,duration_type,min_value,max_value,min_duration,max_duration,negative_event_value,negative_duration
0,DOUBLE,DOUBLE,10.01,4999.98,5.0,300.0,0,0


## 10. `event_date`/`process_date` -- cast correctness, rollback check, window check

Given `satisfaction_surveys` showed a notably higher rollback rate (79.5%) than `transactions`/`call_center_interactions` (~25-33%) with no hour-of-day explanation captured, checking hour-of-day here directly rather than leaving it open again.

In [57]:
q("""
    SELECT
        typeof(event_date) AS event_date_type, typeof(process_date) AS process_date_type,
        min(event_date) AS earliest_event, max(event_date) AS latest_event,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) AS total,
        count(*) FILTER (WHERE process_date <> CAST(event_date AS DATE)) AS process_date_differs
    FROM silver.fact_digital_events
    GROUP BY ALL
""")

,event_date_type,process_date_type,earliest_event,latest_event,earliest_process,latest_process,total,process_date_differs
0,TIMESTAMP,DATE,2023-06-17 06:02:03,2026-06-18 06:04:05,2023-06-17,2026-06-17,15620994,3930816


In [58]:
# Hour-of-day cross-tab for rolled-back rows, same diagnostic used on transactions.
q("""
    SELECT
        EXTRACT(HOUR FROM event_date) AS hour_of_day,
        count(*) FILTER (WHERE process_date = CAST(event_date AS DATE)) AS same_day,
        count(*) FILTER (WHERE process_date <> CAST(event_date AS DATE)) AS rolled_back
    FROM silver.fact_digital_events
    GROUP BY 1
    ORDER BY 1
""")

,hour_of_day,same_day,rolled_back
0,0,0,646937
1,1,0,652578
2,2,0,649799
3,3,0,651541
4,4,0,651193
5,5,0,650693
6,6,626336,28075
7,7,645510,0
8,8,649416,0
9,9,650427,0


In [59]:
q("""
    SELECT
        count(*) FILTER (WHERE event_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE event_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_digital_events
""")

,before_window,after_window
0,0,2060


## 11. `utm_source`/`utm_medium`/`utm_campaign` -- do they populate together?

In [60]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE utm_source IS NOT NULL) AS has_source,
        count(*) FILTER (WHERE utm_medium IS NOT NULL) AS has_medium,
        count(*) FILTER (WHERE utm_campaign IS NOT NULL) AS has_campaign,
        count(*) FILTER (WHERE utm_source IS NOT NULL AND utm_medium IS NOT NULL AND utm_campaign IS NOT NULL) AS all_three,
        count(*) FILTER (WHERE (utm_source IS NOT NULL OR utm_medium IS NOT NULL OR utm_campaign IS NOT NULL)
                          AND NOT (utm_source IS NOT NULL AND utm_medium IS NOT NULL AND utm_campaign IS NOT NULL)) AS partial_only
    FROM silver.fact_digital_events
""")

,total,has_source,has_medium,has_campaign,all_three,partial_only
0,15620994,839045,839134,838917,757411,125565


## 12. `session_id` -- cardinality, events-per-session sanity

In [61]:
q("""
    SELECT count(DISTINCT session_id) AS distinct_sessions, count(*) AS total_events,
           round(count(*) * 1.0 / count(DISTINCT session_id), 2) AS avg_events_per_session
    FROM silver.fact_digital_events
""")

,distinct_sessions,total_events,avg_events_per_session
0,1837415,15620994,8.5


In [62]:
# Distribution of events-per-session, to spot any single outlier session dominating the table.
q("""
    WITH per_session AS (
        SELECT session_id, count(*) AS n FROM silver.fact_digital_events GROUP BY 1
    )
    SELECT min(n) AS min_events, max(n) AS max_events,
           median(n) AS median_events, avg(n) AS avg_events
    FROM per_session
""")

,min_events,max_events,median_events,avg_events
0,2,15,9.0,8.501614


## 13. `ip_address`/`page_url`/`element_id`/`action` -- light format sanity on free-text columns

Not full validation, just checking these look like what their names claim (populated sample, no obviously broken format) since these are free-text and don't have a fixed domain to check against.

In [63]:
display(q_full("""
    SELECT ip_address, page_url, element_id, action
    FROM bronze.digital_events
    WHERE ip_address IS NOT NULL AND page_url IS NOT NULL AND element_id IS NOT NULL AND action IS NOT NULL
    LIMIT 15
"""))
print()
print("ip_address looking like a valid IPv4 (simple regex check):")
display(q("""
    SELECT count(*) AS total,
           count(*) FILTER (WHERE regexp_matches(ip_address, '^\\d{1,3}\\.\\d{1,3}\\.\\d{1,3}\\.\\d{1,3}$')) AS looks_like_ipv4
    FROM bronze.digital_events WHERE ip_address IS NOT NULL
"""))

,ip_address,page_url,element_id,action
0,142.150.185.148,/payments,payment_form,initiate_payment
1,142.150.185.148,/login,login_form,login
2,155.107.175.53,/help,help_page,view_help
3,155.107.175.53,/home,home_page,view_home
4,155.107.175.53,/transactions,transactions_page,view_transactions
5,155.107.175.53,/help,help_page,view_help
6,155.107.175.53,/payments,payment_form,initiate_payment
7,155.107.175.53,/products/loans,loans_product,view_product
8,155.107.175.53,/products,products_page,view_products
9,155.107.175.53,/transactions,transactions_page,view_transactions



ip_address looking like a valid IPv4 (simple regex check):


,total,looks_like_ipv4
0,14840139,14840139


## Checklist

- [ ] Column inventory (1): all 25 Silver columns present, types correct
- [ ] Row reconciliation (2): resolves dictionary's 10M vs. `table_specs.py`'s 15.6M comment against the real count, dedup clean
- [ ] NOT NULL audit (3): all 8 declared columns populated -- note `is_mobile` fix candidate
- [ ] Nullable columns (4): population rates make sense for all 17 nullable columns
- [ ] Domain checks (5): real `event_type`/`event_category`/`channel`/`platform` values captured directly (dictionary text was truncated)
- [ ] FK integrity (6): both FKs (`customer_id`, `product_id`) reproduced directly
- [ ] `ip_country` (7): canonicalization behaves the same way as `transaction_country`
- [ ] `is_mobile` (8): boolean sanity, lines up with `channel` the way you'd expect
- [ ] Numeric ranges (9): `event_value`/`duration_seconds` no negatives
- [ ] Dates (10): rollback rate explained via hour-of-day, window check
- [ ] UTM columns (11): populate together or independently
- [ ] `session_id` (12): cardinality and events-per-session sane, no runaway outlier
- [ ] Free-text sanity (13): `ip_address`/`page_url`/`element_id`/`action` look like what they claim to be

In [64]:
con.close()
print("closed")

closed
